# IT3051 – Telco Customer Churn Prediction
## Data Preprocessing

**Member 3 – Encoding, Scaling & Train/Test Preprocessing Pipeline**

This notebook prepares the feature-selected dataset for machine learning.

Main tasks:
- Load the feature-selected dataset
- Separate the target variable from the input features
- Convert the target into 0 and 1
- Split the data into training and testing sets
- Encode categorical features
- Scale numerical features
- Build a preprocessing pipeline
- Prevent data leakage
- Save the preprocessing pipeline

In [1]:
# Import the libraries needed for preprocessing

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## 1. Load the Feature-Selected Dataset

The feature-selected dataset was prepared by Member 2.

It contains the features that were selected after removing identifiers, leakage variables and unnecessary features.

This dataset is the starting point for my preprocessing work.

In [2]:
# Load the dataset created by Member 2

df = pd.read_csv("feature_selected_data.csv")

print("Dataset shape:", df.shape)

Dataset shape: (7043, 30)


## 2. Dataset Inspection

Before preprocessing, I first check the size, columns, data types and missing values in the dataset.

This helps confirm that the dataset was loaded correctly and shows what type of preprocessing is needed.

In [8]:
# Check the dataset

print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (7043, 30)

First 5 rows:


,Gender,Age,Married,Number of Dependents,Number of Referrals,Tenure in Months,Offer,Phone Service,Avg Monthly Long Distance Charges,Multiple Lines,...,Contract,Paperless Billing,Payment Method,Monthly Charge,Total Refunds,Total Extra Data Charges,Churn Label,CLTV,Total Services,Tenure Group
0,Male,78,No,0,0,1,No Offer,No,0.00,No,...,Month-to-Month,Yes,Bank Withdrawal,39.65,0.00,20,Yes,5433,3,0-12
1,Female,74,Yes,1,1,8,Offer E,Yes,48.85,Yes,...,Month-to-Month,Yes,Credit Card,80.65,0.00,0,Yes,5302,5,0-12
2,Male,71,No,3,0,18,Offer D,Yes,11.33,Yes,...,Month-to-Month,Yes,Bank Withdrawal,95.45,45.61,0,Yes,3179,7,13-24
3,Female,78,Yes,1,1,25,Offer C,Yes,19.76,No,...,Month-to-Month,Yes,Bank Withdrawal,98.50,13.43,0,Yes,5337,7,25-48
4,Female,80,Yes,1,1,37,Offer C,Yes,6.33,Yes,...,Month-to-Month,Yes,Bank Withdrawal,76.50,0.00,0,Yes,2793,4,25-48



Data types:
Gender                                object
Age                                    int64
Married                               object
Number of Dependents                   int64
Number of Referrals                    int64
Tenure in Months                       int64
Offer                                 object
Phone Service                         object
Avg Monthly Long Distance Charges    float64
Multiple Lines                        object
Internet Type                         object
Avg Monthly GB Download                int64
Online Security                       object
Online Backup                         object
Device Protection Plan                object
Premium Tech Support                  object
Streaming TV                          object
Streaming Movies                      object
Streaming Music                       object
Unlimited Data                        object
Contract                              object
Paperless Billing                     obje

## 3. Separate Target and Predictor Variables

The target variable is **Churn Label**.

It is the value that the machine learning model will try to predict.

All other selected columns are used as predictor variables.

In [9]:
# Separate the target from the input features

X = df.drop("Churn Label", axis=1)
y = df["Churn Label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 29)
y shape: (7043,)


## 4. Convert the Target Variable

The original Churn Label contains two values:

- No = customer did not churn
- Yes = customer churned

For machine learning, these are converted into numerical values:

- 0 = No Churn
- 1 = Churn

In [10]:
# Convert the target from Yes/No to 1/0

y = y.map({
    "No": 0,
    "Yes": 1
})

print("Target values:")
print(y.value_counts())

Target values:
0    5174
1    1869
Name: Churn Label, dtype: int64


In [11]:
print("\nTarget distribution:")
print((y.value_counts(normalize=True) * 100).round(2))


Target distribution:
0    73.46
1    26.54
Name: Churn Label, dtype: float64


## 5. Train/Test Split

The dataset is divided into two parts:

- **Training data (80%)** – used to train the machine learning model.
- **Testing data (20%)** – used later to evaluate the model.

Stratification is used so that both datasets keep approximately the same proportion of churned and non-churned customers.

The test data must remain separate so that it can provide an unbiased evaluation later.

In [12]:
# Split the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 29)
X_test : (1409, 29)
y_train: (5634,)
y_test : (1409,)


## 6. Check the Class Distribution

After splitting the data, I check the target distribution in both the training and testing sets.

The percentages should be approximately similar because stratification was used.

In [13]:
# Check the class distribution in both sets

print("Training target distribution:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting target distribution:")
print((y_test.value_counts(normalize=True) * 100).round(2))

Training target distribution:
0    73.46
1    26.54
Name: Churn Label, dtype: float64

Testing target distribution:
0    73.46
1    26.54
Name: Churn Label, dtype: float64


## 7. Numerical Features

Numerical features contain values that represent quantities, such as age, tenure and monthly charges.

These features will be:
1. Checked for missing values
2. Filled using the median if necessary
3. Scaled using StandardScaler

Scaling puts numerical features on a similar scale so that features with larger values do not dominate smaller-valued features.

In [14]:
# List the numerical features selected by Member 2

numeric_features = [
    "Age",
    "Number of Dependents",
    "Number of Referrals",
    "Tenure in Months",
    "Avg Monthly Long Distance Charges",
    "Avg Monthly GB Download",
    "Monthly Charge",
    "Total Refunds",
    "Total Extra Data Charges",
    "CLTV",
    "Total Services"
]

print("Number of numerical features:", len(numeric_features))
print(numeric_features)

Number of numerical features: 11
['Age', 'Number of Dependents', 'Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download', 'Monthly Charge', 'Total Refunds', 'Total Extra Data Charges', 'CLTV', 'Total Services']


## 8. Categorical Features

Categorical features contain groups or labels instead of continuous numbers.

Examples include Gender, Contract, Internet Type and Payment Method.

These values cannot be directly given to most machine learning algorithms, so they will be converted into numerical values using One-Hot Encoding.

In [15]:
# List the categorical features selected by Member 2

categorical_features = [
    "Gender",
    "Married",
    "Offer",
    "Phone Service",
    "Multiple Lines",
    "Internet Type",
    "Online Security",
    "Online Backup",
    "Device Protection Plan",
    "Premium Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Streaming Music",
    "Unlimited Data",
    "Contract",
    "Paperless Billing",
    "Payment Method",
    "Tenure Group"
]

print("Number of categorical features:", len(categorical_features))
print(categorical_features)

Number of categorical features: 18
['Gender', 'Married', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Tenure Group']


## 9. Feature List Validation

Before building the preprocessing pipeline, I check that all predictor columns are included in either the numerical or categorical feature lists.

This helps prevent accidentally leaving a feature out of the preprocessing pipeline.

In [16]:
# Combine both feature lists

all_features = numeric_features + categorical_features

print("Total features in lists:", len(all_features))

print("\nFeatures missing from lists:")
print(set(X.columns) - set(all_features))

print("\nFeatures not found in dataset:")
print(set(all_features) - set(X.columns))

Total features in lists: 29

Features missing from lists:
set()

Features not found in dataset:
set()


## 10. Numerical Preprocessing

Numerical features are processed using two steps:

1. **Median Imputation** – fills missing numerical values using the median.
2. **StandardScaler** – standardizes the numerical values.

The median is used because it is less affected by extreme values than the mean.

In [17]:
# Preprocessing steps for numerical features

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

## 11. Categorical Preprocessing

Categorical features are processed using two steps:

1. **Most Frequent Imputation** – fills missing categorical values using the most common value.
2. **One-Hot Encoding** – converts categories into numerical columns.

`handle_unknown="ignore"` allows the pipeline to handle a category that was not seen during training.

In [18]:
# Preprocessing steps for categorical features

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

## 12. Combined Preprocessing Pipeline

Numerical and categorical features require different preprocessing methods.

`ColumnTransformer` allows us to apply the correct method to each group of features.

Numerical features are imputed and scaled, while categorical features are imputed and one-hot encoded.

                  Dataset
                     │
          ┌──────────┴──────────┐
          ↓                     ↓
     Numerical              Categorical
       features                features
          ↓                     ↓
      Imputation            Imputation
          ↓                     ↓
       Scaling           One-Hot Encoding
          └──────────┬──────────┘
                     ↓
            Processed Dataset

In [19]:
# Apply the correct preprocessing to each feature type

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## 13. Fit the Preprocessing Pipeline on Training Data

The preprocessing pipeline is fitted only on the training data.

This is important because the test data must remain unseen during preprocessing.

For example, the scaler learns its mean and standard deviation from the training data only.

This prevents data leakage from the test set.

In [20]:
# Learn preprocessing rules from training data

X_train_processed = preprocessor.fit_transform(X_train)

print("Processed training shape:", X_train_processed.shape)

Processed training shape: (5634, 57)


## 14. Transform the Test Data

The already-fitted preprocessing pipeline is now used to transform the test data.

The pipeline is not fitted again on the test data.

This keeps the test data completely separate from the learning process.

In [21]:
# Apply the already-fitted preprocessing to the test data

X_test_processed = preprocessor.transform(X_test)

print("Processed testing shape:", X_test_processed.shape)

Processed testing shape: (1409, 57)


## 15. Final Shape Check

The final shapes are checked to make sure that the training and testing data were processed correctly.

The number of processed features should be the same for both training and testing data.

In [22]:
# Check the original and processed data sizes

print("Original X:", X.shape)

print("\nBefore preprocessing:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nAfter preprocessing:")
print("X_train_processed:", X_train_processed.shape)
print("X_test_processed :", X_test_processed.shape)

print("\ny_train:", y_train.shape)
print("y_test :", y_test.shape)

Original X: (7043, 29)

Before preprocessing:
X_train: (5634, 29)
X_test : (1409, 29)

After preprocessing:
X_train_processed: (5634, 57)
X_test_processed : (1409, 57)

y_train: (5634,)
y_test : (1409,)


## 16. Processed Feature Names

One-Hot Encoding creates additional columns from categorical variables.

This step displays the names of the final processed features so that we can understand what the machine learning model will receive.

In [23]:
# Get the names of the processed features

feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

print("\nFirst 20 processed features:")
print(feature_names[:20])

Number of processed features: 57

First 20 processed features:
['num__Age' 'num__Number of Dependents' 'num__Number of Referrals'
 'num__Tenure in Months' 'num__Avg Monthly Long Distance Charges'
 'num__Avg Monthly GB Download' 'num__Monthly Charge' 'num__Total Refunds'
 'num__Total Extra Data Charges' 'num__CLTV' 'num__Total Services'
 'cat__Gender_Female' 'cat__Gender_Male' 'cat__Married_No'
 'cat__Married_Yes' 'cat__Offer_No Offer' 'cat__Offer_Offer A'
 'cat__Offer_Offer B' 'cat__Offer_Offer C' 'cat__Offer_Offer D']


## 17. Check Numerical Scaling

StandardScaler changes the numerical features so that they are on a comparable scale.

For the training data, the transformed numerical features should have values centred around zero.

In [24]:
# Check a few processed training values

print(X_train_processed[:5])

[[-0.21136583 -0.48381926 -0.65257962 -0.67032436 -1.11469386 -0.80975211
  -0.31834335 -0.24968348 -0.2709701  -0.52995901 -0.06760229  0.
   1.          1.          0.          1.          0.          0.
   0.          0.          0.          0.          1.          0.
   1.          0.          1.          0.          0.          1.
   0.          0.          1.          1.          0.          1.
   0.          1.          0.          1.          0.          1.
   0.          0.          1.          1.          0.          0.
   1.          0.          1.          0.          0.          0.
   1.          0.          0.        ]
 [ 0.91936212  1.5835586  -0.65257962 -0.83274861 -0.39033802 -0.12346488
   1.34124599 -0.24968348 -0.2709701   1.2694606   1.3161263   1.
   0.          1.          0.          1.          0.          0.
   0.          0.          0.          0.          1.          0.
   1.          0.          0.          1.          0.          1.
   0.          0.    

## 18. Save the Preprocessing Pipeline

The preprocessing pipeline contains the steps used to prepare the data.

We save it so the same preprocessing steps can be reused later when building and testing the machine learning model.


In [25]:
# Save the preprocessing pipeline
joblib.dump(
    preprocessor,
    "models/preprocessing_pipeline.pkl"
)

print("Preprocessing pipeline saved successfully.")

Preprocessing pipeline saved successfully.


## 19. Check the Saved Pipeline

This step checks whether the preprocessing pipeline was saved correctly.

The saved pipeline can be loaded again and reused later.

In [26]:
# Load the saved pipeline
saved_preprocessor = joblib.load(
    "models/preprocessing_pipeline.pkl"
)

print("Pipeline loaded successfully.")

Pipeline loaded successfully.


## 20. Save the Processed Training and Testing Data

The processed training and testing data are saved for the next stage of the project.

The training data will be used to build the machine learning models, while the testing data will be used to evaluate them.

In [27]:
# Convert processed data to DataFrame
X_train_processed_df = pd.DataFrame(
    X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed,
    columns=feature_names
)

X_test_processed_df = pd.DataFrame(
    X_test_processed.toarray() if hasattr(X_test_processed, "toarray") else X_test_processed,
    columns=feature_names
)

# Add target column
X_train_processed_df["Churn Label"] = y_train.values
X_test_processed_df["Churn Label"] = y_test.values

# Save the processed datasets
X_train_processed_df.to_csv(
    "train_processed.csv",
    index=False
)

X_test_processed_df.to_csv(
    "test_processed.csv",
    index=False
)

print("Processed training and testing data saved successfully.")

Processed training and testing data saved successfully.


## 21. Final Check

This step checks the final processed datasets.

It confirms the number of rows and features and shows the distribution of the target variable.

In [28]:
# Check final dataset shapes
print("Training data shape:", X_train_processed_df.shape)
print("Testing data shape :", X_test_processed_df.shape)

# Check target distribution
print("\nTraining target:")
print(y_train.value_counts())

print("\nTesting target:")
print(y_test.value_counts())

Training data shape: (5634, 58)
Testing data shape : (1409, 58)

Training target:
0    4139
1    1495
Name: Churn Label, dtype: int64

Testing target:
0    1035
1     374
Name: Churn Label, dtype: int64


## 22. Member 3 Deliverables — Summary

### Final Preprocessing Steps

| Step | What was done |
|---|---|
| Target Encoding | Converted `Churn Label` from `Yes/No` to `1/0` |
| Train/Test Split | Divided the data into 80% training and 20% testing |
| Stratification | Kept the churn class proportion similar in both sets |
| Numerical Preprocessing | Handled missing values and applied `StandardScaler` |
| Categorical Preprocessing | Handled missing values and applied `OneHotEncoder` |
| Preprocessing Pipeline | Combined all preprocessing steps using `ColumnTransformer` |
| Data Leakage Prevention | Fitted the preprocessing only on the training data |
| Saved Pipeline | `models/preprocessing_pipeline.pkl` |
| Saved Data | `train_processed.csv` and `test_processed.csv` |

### Handover to Model Development

The following files are provided for the next stage:

- `train_processed.csv` — processed training data.
- `test_processed.csv` — processed testing data.
- `models/preprocessing_pipeline.pkl` — saved preprocessing pipeline.

The target variable is `Churn Label`, where `0 = No Churn` and `1 = Churn`.

The processed data is ready for machine learning model development and evaluation.

## 23. Viva Preparation — Member 3

### What is encoding?

Encoding converts categorical values such as `Yes`, `No`, and `Month-to-month` into numerical values that machine learning models can understand.

### Why use One-Hot Encoding?

It converts categories into separate binary columns without creating an incorrect order between categories.

### What is scaling?

Scaling puts numerical features on a similar scale so that features with larger values do not dominate smaller-valued features.

### Why use StandardScaler?

StandardScaler standardizes numerical features using their mean and standard deviation.

### Why split the data into training and testing sets?

The training data is used to build the model, while the testing data is used to check how well the model works on unseen data.

### Why use `stratify=y`?

It keeps the proportion of churn and non-churn customers similar in both training and testing data.

### Why fit the preprocessing only on training data?

To prevent information from the test data from affecting the preprocessing. This helps prevent data leakage.

### Why use a pipeline?

A pipeline keeps the preprocessing steps together and allows the same steps to be reused later.

### What are the final outputs?

The final outputs are the processed training data, processed testing data, and the saved preprocessing pipeline.

## 24. My Contribution

My responsibility was to prepare the selected features for machine learning.

I performed target encoding, train/test splitting, numerical scaling, categorical encoding, and created a reusable preprocessing pipeline. I also saved the processed datasets and preprocessing pipeline for the model development stage.